In [1]:
!pip install ramanujantools==0.0.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 46.6 MB/s eta 0:00:00


In [2]:
import sympy as sp
from sympy.abc import n
from sympy.physics.quantum import TensorProduct

from ramanujantools import Matrix, LinearRecurrence
from ramanujantools.pcf import PCF

In [3]:
# The e PCF
pcf_e = PCF(n, n)
iv_e = Matrix([[1, 1], [1, 0]])
display(pcf_e.recurrence.recurrence_matrix)

Matrix([
[0, n],
[1, n]])

In [4]:
# The pi PCF
pcf_pi = PCF(2*n + 1, n**2)
iv_pi = Matrix([[0, 4], [1, 1]])
display(pcf_pi.recurrence.recurrence_matrix)

Matrix([
[0,    n**2],
[1, 2*n + 1]])

In [5]:
# The tensor product
m = TensorProduct(pcf_e.recurrence.recurrence_matrix, pcf_pi.recurrence.recurrence_matrix)
display(m)

Matrix([
[0,       0, 0,        n**3],
[0,       0, n, n*(2*n + 1)],
[0,    n**2, 0,        n**3],
[1, 2*n + 1, n, n*(2*n + 1)]])

In [6]:
# The coboundary/gauge matrix U to convert the tensor product matrix M(n) into a companion form matrix
U = m.companion_coboundary_matrix()
display(U)

Matrix([
[1, 0,      n*n**2,             (n + 1)*(2*n**4 + 3*n**3)],
[0, 0, n*(2*n + 1),      (n + 1)*(5*n**3 + 10*n**2 + 4*n)],
[0, 0,      n*n**2,    (n + 1)*(2*n**4 + 5*n**3 + 3*n**2)],
[0, 1, n*(2*n + 1), (n + 1)*(5*n**3 + 15*n**2 + 14*n + 4)]])

In [7]:
# The gauged matrix C, in companion form
C = m.coboundary(U)
display(C)

Matrix([
[0, 0, 0,                  n**3*(-n - 2)*(n + 1)**2*(n**3 + n**2 - 8*n - 11)/(n**3 - 2*n**2 - 7*n - 3)],
[1, 0, 0,       (n + 1)**2*(n + 2)*(2*n**5 + 3*n**4 - 13*n**3 - 21*n**2 + 4)/(n**3 - 2*n**2 - 7*n - 3)],
[0, 1, 0, (n + 2)*(n**6 + 9*n**5 + 8*n**4 - 87*n**3 - 249*n**2 - 234*n - 68)/(n**3 - 2*n**2 - 7*n - 3)],
[0, 0, 1,                      (n + 2)*(2*n**4 + n**3 - 26*n**2 - 48*n - 19)/(n**3 - 2*n**2 - 7*n - 3)]])

In [8]:
# The equivalent recurrence
r = LinearRecurrence(C)
r

LinearRecurrence([-n**3 + 2*n**2 + 7*n + 3, 2*n**5 + 5*n**4 - 24*n**3 - 100*n**2 - 115*n - 38, n**7 + 11*n**6 + 26*n**5 - 71*n**4 - 423*n**3 - 732*n**2 - 536*n - 136, 2*n**8 + 11*n**7 + 9*n**6 - 54*n**5 - 143*n**4 - 127*n**3 - 26*n**2 + 20*n + 8, -n**9 - 5*n**8 - n**7 + 36*n**6 + 82*n**5 + 71*n**4 + 22*n**3])

In [9]:
# The multiplication of each combination of initial values of e and pi, using the same tensor product
iv_crossed = TensorProduct(iv_e, iv_pi)
display(iv_crossed)

Matrix([
[0, 4, 0, 4],
[1, 1, 1, 1],
[0, 4, 0, 0],
[1, 1, 0, 0]])

In [10]:
# Finding the gauged initial values for pi + e - taking the sum of the two middle rows, over the last row.
iv_non_gauged = Matrix.vstack(iv_crossed.row(1) + iv_crossed.row(2), iv_crossed.row(3))
iv = iv_non_gauged * U.subs({n: 1})
iv

Matrix([
[1, 1, 20, 296],
[1, 0,  4,  48]])

In [11]:
# Sanity check: numerically testing the convergence of the constructed recurrence
l = r.limit(100, 1, iv)
l.as_float() - (l.mp.pi + l.mp.e)

mpf('7.30973767934817978491403764441673720298264334053641379951778642879617637643979365790001e-79')